# **North Sea temperature data (Jan. 1960 - Jun. 2024)**

In [1]:
library(BioExtremeEvent)
library(car)
library(dplyr)
library(effects)
library(energy)
library(foreach)
library(furrr)
library(future)
library(ggplot2)
library(gifski)
library(gsl)
library(magick)
library(marmap)
library(metR)
library(ncdf4)
library(parallel)
library(patchwork)
library(purrr)
library(RColorBrewer)
library(readr)
library(rprojroot)
library(terra)
library(tidyr)
library(tidyterra)

Loading required package: carData

Attaching package: ‘dplyr’

The following object is masked from ‘package:car’:

    recode

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

lattice theme set by effectsTheme()
See ?effectsTheme for details.
Loading required package: future
Linking to ImageMagick 6.9.12.98
Enabled features: fontconfig, freetype, fftw, heic, lcms, pango, raw, webp, x11
Disabled features: cairo, ghostscript, rsvg
Using 16 threads
Registered S3 methods overwritten by 'adehabitatMA':
  method                       from
  print.SpatialPixelsDataFrame sp  
  print.SpatialPixels          sp  

Attaching package: ‘marmap’

The following object is masked from ‘package:grDevices’:

    as.raster


Attaching package: ‘purrr’

The following object is masked from ‘package:metR’:

    cross

The following objects are masked from ‘package:foreach’:

    accumulate,

In [ ]:
proj_root <- find_root(has_file(".git") | is_rstudio_project)
setwd(file.path(proj_root, "SR_Estimation", "stat_analysis_2026", "data"))

[1] "/home/mathis/NorthSeaSR_Estimation/SR_Estimation/stat_analysis_2026/data"

# Download and import the data

All of the data used in this section is exacted from Copernicus Marine. The web link from which the data was downloaded in the following : <https://data.marine.copernicus.eu/product/INSITU_GLO_PHY_TS_OA_MY_013_052/description>.  

## Sea water temperature

In [ ]:
# BioExtremeEvent::BEE.load.copernicus(
#     username = "username", # In addition of the API copy&paste
#     password = "password", # In addition of the API copy&paste
#     dataset_id="cmems_obs-ins_glo_phy-temp-sal_my_cora-oa_P1M",
#     dataset_version="202411",
#     variables="TEMP",
#     minimum_longitude=-4.5,
#     maximum_longitude=13.5,
#     minimum_latitude=48.5,
#     maximum_latitude=62.5,
#     start_datetime="1960-01-01T00:00:00",
#     end_datetime="2024-06-01T00:00:00",
#     coordinates_selection_method = "strict-inside",
#     disable_progress_bar = FALSE,
#     output_directory = here::here("pathway_to_save") # In addition of the API copy&paste
# )

In [ ]:
copernicus_data_temp <- terra::rast("cmems_obs-ins_glo_phy-temp-sal_my_cora-oa_P1M_TEMP_4.50W-13.50E_48.60N-62.50N_1.00-5500.00m_1960-01-01-2024-06-01.nc")

In [ ]:
# --- Investigate the data ---
print(copernicus_data_temp)
cat("\n")

# --- Dimensions ---
cat("Number of layers  :", nlyr(copernicus_data_temp), "\n")
cat("Resolution        :", res(copernicus_data_temp), "\n")
cat("GPS coordinates   :", as.character(ext(copernicus_data_temp)), "\n")
cat("\n")

# --- Available depths ---
cat("Available depths :\n")
print(head(depth(copernicus_data_temp)))
cat("\n")

# --- Available dates ---
cat("Available dates :\n")
print(head(time(copernicus_data_temp)))
cat("\n")

# --- Time range ---
dates <- time(copernicus_data_temp)
cat("First date :", as.character(min(dates)), "\n")
cat("Last date :", as.character(max(dates)), "\n")
cat("\n")

In [ ]:
# Targeted depths
target_depths <- c(1, 10, 20, 50, 100, 200, 300, 400, 500, 600, 700)

# Unique targeted depths
available_depths <- unique(depth(copernicus_data_temp))

# Indices of availables vs targeted depths
depth_indices <- sapply(target_depths, function(d) which.min(abs(available_depths - d)))

data.frame(target = target_depths,
           available = available_depths[depth_indices],
           indice = depth_indices)

In [ ]:
# Download bathymetry
# bathy <- getNOAA.bathy(
#   lon1 = -4.75, lon2 = 13.75,
#   lat1 = 48.55, lat2 = 62.55,
#   resolution = 1,  # 1 minute d'arc (~1.8 km)
#   keep = TRUE)

In [ ]:
# --- Parameters ---
plot_date <- "1960-01"
n_depths  <- length(available_depths)   

bathy <- read.bathy("marmap_coord_-4.75;48.55;13.75;62.55_1.csv", header = TRUE)
bathy_df <- fortify(bathy)
colnames(bathy_df) <- c("lon", "lat", "depth")
bathy_df$depth <- ifelse(bathy_df$depth > 0, NA, bathy_df$depth)

iso_levels <- c(-1, -10, -20, -50, -100, -200, -300, -400, -500, -600, -700)
pal <- rev(RColorBrewer::brewer.pal(11, "RdYlBu"))
all_dates <- format(unique(time(copernicus_data_temp)), "%Y-%m")
t <- which(all_dates == plot_date)

In [ ]:
# # --- Fonction to check individual plots ---
# make_plot <- function(i) {
#   
#   # Profondeur × timestep
#   layer_index <- (t - 1) * n_depths + depth_indices[i]
#   layer       <- copernicus_data_temp[[layer_index]]
#   d           <- target_depths[i]
#   
#   p <- ggplot() +
#     geom_spatraster(data = layer) +
#     scale_fill_gradientn(colors   = pal,
#                          na.value = "grey90",
#                          name     = "°C",
#                          limits   = c(0, 25)) +
#     annotation_borders("world", colour = "black", linewidth = 0.3) +
#     coord_sf(xlim = c(-4.75, 13.75), ylim = c(48.55, 62.55))
#   
#   p <- p +
#     geom_contour(data      = bathy_df,
#                  aes(x = lon, y = lat, z = depth),
#                  breaks    = -d,
#                  na.rm     = TRUE,
#                  color     = "red",
#                  linewidth = 0.6)
#   
#   p +
#     labs(title    = paste0(d, " m"),
#          subtitle = paste0("Isobathe ", d, "m"),
#          x = NULL, y = NULL) +
#     theme_minimal(base_size = 11) +
#     theme(legend.position = "right",
#           plot.title    = element_text(face = "bold", hjust = 0.5),
#           plot.subtitle = element_text(hjust = 0.5, color = "grey50"))
# }
# 
# # --- Path to save the plots ---
# output_dir <- "/home/mathis/Desktop/temp_per_depth"
# dir.create(output_dir, showWarnings = FALSE)
# 
# # --- Global map ---
# plots    <- lapply(seq_along(target_depths), make_plot)
# p_global <- wrap_plots(plots, ncol = 3) +
#   plot_annotation(
#     title    = "Temperature per depth — Jan 1960",
#     subtitle = "Source : Copernicus CORA | Isobaths : ETOPO via marmap",
#     theme    = theme(plot.title = element_text(size = 14, face = "bold"))
#   )
# 
# ggsave(file.path(output_dir, paste0("temp_overview_", plot_date, ".png")),
#        plot = p_global, width = 14, height = 16, dpi = 150)
# 
# # --- Individual map ---
# for (i in seq_along(target_depths)) {
#   p_ind <- make_plot(i)
#   ggsave(file.path(output_dir, paste0("temp_depth_", target_depths[i], "m_", plot_date, ".png")),
#          plot = p_ind, width = 8, height = 7, dpi = 150)
# }
# 
# cat("Plots saved in temp_per_depth folder\n")

## Error on temperature (% variance)

<https://data.marine.copernicus.eu/product/INSITU_GLO_PHY_TS_OA_MY_013_052/description>

In [ ]:
# BioExtremeEvent::BEE.load.copernicus(
#     username = "username", # In addition of the API copy&paste
#     password = "password", # In addition of the API copy&paste
#     dataset_id="cmems_obs-ins_glo_phy-temp-sal_my_cora-oa_P1M",
#     dataset_version="202411",
#     variables="TEMP_PCTVAR",
#     minimum_longitude=-4.5,
#     maximum_longitude=13.5,
#     minimum_latitude=48.5,
#     maximum_latitude=62.5,
#     start_datetime="1960-01-01T00:00:00",
#     end_datetime="2024-06-01T00:00:00",
#     coordinates_selection_method = "strict-inside",
#     disable_progress_bar = FALSE,
#     output_directory = here::here("path_to_save") # In addition of the API copy&paste
# )

In [ ]:
copernicus_data_temp_var <- terra::rast("cmems_obs-ins_glo_phy-temp-sal_my_cora-oa_P1M_TEMP_PCTVAR_4.50W-13.50E_48.60N-62.50N_1.00-5500.00m_1960-01-01-2024-06-01.nc")

In [ ]:
# --- Investigate the data ---
print(copernicus_data_temp_var)
cat("\n")

# --- Dimensions ---
cat("Number of layers  :", nlyr(copernicus_data_temp_var), "\n")
cat("Resolution        :", res(copernicus_data_temp_var), "\n")
cat("GPS coordinates   :", as.character(ext(copernicus_data_temp_var)), "\n")
cat("\n")

# --- Available depths ---
cat("Available depths :\n")
print(head(depth(copernicus_data_temp_var)))
cat("\n")

# --- Available dates ---
cat("Available dates :\n")
print(head(time(copernicus_data_temp_var)))
cat("\n")

# --- Time range ---
dates <- time(copernicus_data_temp_var)
cat("First date :", as.character(min(dates)), "\n")
cat("Last date :", as.character(max(dates)), "\n")
cat("\n")

In [ ]:
# Targeted depths
target_depths <- c(1, 10, 20, 50, 100, 200, 300, 400, 500, 600, 700)

# Available depth
available_depths <- unique(depth(copernicus_data_temp_var))

# Unique available vs targeted depths
depth_indices <- sapply(target_depths, function(d) which.min(abs(available_depths - d)))

data.frame(target = target_depths,
           available = available_depths[depth_indices],
           indice = depth_indices)

In [ ]:
# Download bathy
# bathy <- getNOAA.bathy(
#   lon1 = -4.75, lon2 = 13.75,
#   lat1 = 48.55, lat2 = 62.55,
#   resolution = 1,  # 1 minute d'arc
#   keep = TRUE)

In [ ]:
# --- Parameters ---
plot_date <- "1960-01"
n_depths  <- length(available_depths)   

bathy <- read.bathy("marmap_coord_-4.75;48.55;13.75;62.55_1.csv", header = TRUE)
bathy_df <- fortify(bathy)
colnames(bathy_df) <- c("lon", "lat", "depth")
bathy_df$depth <- ifelse(bathy_df$depth > 0, NA, bathy_df$depth)

iso_levels <- c(-1, -10, -20, -50, -100, -200, -300, -400, -500, -600, -700)
pal <- rev(RColorBrewer::brewer.pal(11, "RdYlBu"))
all_dates <- format(unique(time(copernicus_data_temp)), "%Y-%m")
t <- which(all_dates == plot_date)

In [ ]:
# # --- Fonction for individual plots ---
# make_plot <- function(i) {
#   
#   # ← CORRECTION CLEF : indexation (timestep × profondeur)
#   layer_index <- (t - 1) * n_depths + depth_indices[i]
#   layer       <- copernicus_data_temp_var[[layer_index]]
#   d           <- target_depths[i]
#   
#   p <- ggplot() +
#     geom_spatraster(data = layer) +
#     scale_fill_gradientn(colors   = pal,
#                          na.value = "grey90",
#                          name     = "variance %",
#                          limits   = c(0, 100)) +   # ← adapté pour % variance
#     annotation_borders("world", colour = "black", linewidth = 0.3) +
#     coord_sf(xlim = c(-4.75, 13.75), ylim = c(48.55, 62.55))
#   
#   p <- p +
#     geom_contour(data      = bathy_df,
#                  aes(x = lon, y = lat, z = depth),
#                  breaks    = -d,
#                  na.rm     = TRUE,
#                  color     = "red",
#                  linewidth = 0.6)
#   
#   p +
#     labs(title    = paste0(d, " m"),
#          subtitle = paste0("Isobathe ", d, "m"),
#          x = NULL, y = NULL) +
#     theme_minimal(base_size = 11) +
#     theme(legend.position = "right",
#           plot.title    = element_text(face = "bold", hjust = 0.5),
#           plot.subtitle = element_text(hjust = 0.5, color = "grey50"))
# }
# 
# # --- Dossier sauvegarde ---
# output_dir <- "/home/mathis/Desktop/temp_var_per_depth"
# dir.create(output_dir, showWarnings = FALSE)
# 
# # --- Global map ---
# plots    <- lapply(seq_along(target_depths), make_plot)
# p_global <- wrap_plots(plots, ncol = 3) +
#   plot_annotation(
#     title    = "Temperature variance per depth — Jan 1960",
#     subtitle = "Source : Copernicus CORA | Isobaths : ETOPO via marmap",
#     theme    = theme(plot.title = element_text(size = 14, face = "bold"))
#   )
# 
# ggsave(file.path(output_dir, paste0("temp_var_overview_", plot_date, ".png")),
#        plot = p_global, width = 14, height = 16, dpi = 150)
# 
# # --- Individual map ---
# for (i in seq_along(target_depths)) {
#   p_ind <- make_plot(i)
#   ggsave(file.path(output_dir, paste0("temp_var_depth_", target_depths[i], "m_", plot_date, ".png")),
#          plot = p_ind, width = 8, height = 7, dpi = 150)
# }
# 
# cat("Plots saved in temp_var_per_depth folder\n")

# GIF creation

After checking for a single year, we aim at vizualising the whole data : all depths for the whole time range. Then, to make vizualisation easier, a GIF of the time series is made.  This section develops the pipeline to do so.  

## Sea water temperature

**Function to generate the frames**

In [ ]:
generate_temp_frames <- function(depth_target,
                                  nc_path,
                                  output_dir = "/home/mathis/Desktop/temp_gif",
                                  n_workers  = 10,
                                  dpi        = 100) {
  
  all_dates <- format(seq(as.Date("1960-01-01"),
                                 as.Date("2024-06-01"),
                                 by = "month"), "%Y-%m")
  available_depths <- unique(terra::depth(terra::rast(nc_path)))
  n_depths <- length(available_depths)
  depth_i <- which.min(abs(available_depths - depth_target))
  
  cat("Depth investigated :", depth_target, "m → Depth available :",
      available_depths[depth_i], "m\n")
  cat("Number of timesteps :", length(all_dates), "\n")
  
  iso_exists <- any(bathy_df$depth <= -depth_target, na.rm = TRUE)
  frames_dir <- file.path(output_dir, paste0("frames_", depth_target, "m"))
  dir.create(frames_dir, recursive = TRUE, showWarnings = FALSE)
  pal <- rev(RColorBrewer::brewer.pal(11, "RdYlBu"))
  
  make_frame <- function(t_idx, nc_path, n_depths, depth_i, depth_target,
                          bathy_df, pal, all_dates, iso_exists, frames_dir) {
    
    layer_index <- (t_idx - 1) * n_depths + depth_i
    layer       <- terra::rast(nc_path, lyrs = layer_index)
    date_label  <- all_dates[t_idx]
    
    p <- ggplot() +
      geom_spatraster(data = layer) +
      scale_fill_gradientn(colors   = pal,
                           na.value = "grey90",
                           name     = "°C",          
                           limits   = c(-3, 25)) +   
      annotation_borders("world", colour = "black", linewidth = 0.3) +
      coord_sf(xlim = c(-4.75, 13.75), ylim = c(48.55, 62.55))
    
    if (iso_exists) {
      p <- p +
        geom_contour(data      = bathy_df,
                     aes(x = lon, y = lat, z = depth),
                     breaks    = -depth_target,
                     na.rm     = TRUE,
                     color     = "red",
                     linewidth = 0.6)
    }
    
    p <- p +
      labs(title    = paste0("Temperature — ", depth_target, " m"), 
           subtitle = date_label,
           x = NULL, y = NULL) +
      theme_minimal(base_size = 11) +
      theme(legend.position = "right",
            plot.title    = element_text(face = "bold", hjust = 0.5),
            plot.subtitle = element_text(hjust = 0.5, color = "grey50", size = 12))
    
    fname <- file.path(frames_dir, sprintf("frame_%04d_%s.png", t_idx, date_label))
    ggsave(fname, plot = p, width = 8, height = 7, dpi = dpi)
    return(fname)
  }
  
  plan(multisession, workers = n_workers)
  cat("Generating", length(all_dates), "frames on", n_workers, "workers...\n")
  
  frame_files <- future_map_chr(
    seq_along(all_dates),
    ~ make_frame(.x,
                 nc_path      = nc_path,
                 n_depths     = n_depths,
                 depth_i      = depth_i,
                 depth_target = depth_target,
                 bathy_df     = bathy_df,
                 pal          = pal,
                 all_dates    = all_dates,
                 iso_exists   = iso_exists,
                 frames_dir   = frames_dir),
    .progress = TRUE
  )
  
  plan(sequential)
  cat("Frames generated ✓\n")
}

In [ ]:
nc_path_temp <- "cmems_obs-ins_glo_phy-temp-sal_my_cora-oa_P1M_TEMP_4.50W-13.50E_48.60N-62.50N_1.00-5500.00m_1960-01-01-2024-06-01.nc"

for (d in target_depths) {
  cat("\n========================================\n")
  cat("Computing depth :", d, "m\n")
  cat("========================================\n")
  
  generate_temp_frames(depth_target = d, nc_path = nc_path_temp)
  
  plan(sequential)
  gc()
  terra::terraOptions(memfrac = 0.5)
  Sys.sleep(25)
  cat("Memory cleaned ✓\n")
}
cat("\nAll frames are generated ✓\n")

**Function to generate the GIFs (one per depth)**

In [ ]:
generate_gifs_from_frames <- function(base_dir = "/home/mathis/Desktop/temp_gif",
                               fps      = 6) {
  library(gifski)
  
  subdirs <- list.dirs(base_dir, full.names = TRUE, recursive = FALSE)
  subdirs <- subdirs[grepl("frames_\\d+m$", basename(subdirs))]
  
  cat("Subfolders investigated :", length(subdirs), "\n")
  
  for (subdir in subdirs) {
    
    depth_target <- as.integer(gsub("frames_(\\d+)m", "\\1", basename(subdir)))
    cat("\n--- Depth :", depth_target, "m ---\n")
    
    frame_files <- sort(list.files(subdir, pattern = "\\.png$", full.names = TRUE))
    cat("Frames found :", length(frame_files), "\n")
    
    if (length(frame_files) == 0) {
      cat("No frame found, this depth is ignored.\n")
      next
    }
    
    gif_path <- file.path(base_dir,
                           paste0("temp_", depth_target, "m_1960-2024.gif"))
    
    cat("Assembling GIF...\n")
    
    # gifski reads the frames individually — no OOM
    gifski(frame_files,
           gif_file = gif_path,
           delay = 1 / fps,
           loop = 0)        
    
    cat("GIF saved :", gif_path, "\n")
    gc()
  }
  
  cat("\nAll GIFs are generated ✓\n")
}

# --- Calling function ---
generate_gifs_from_frames(base_dir = "/home/mathis/Desktop/temp_gif")

## Error on temperature (% variance)

**Function to create the frames**

In [ ]:
generate_temp_var_frames <- function(depth_target,
                                     nc_path,
                                     output_dir  = "/home/mathis/Desktop/temp_var_gif",
                                     n_workers   = 10,
                                     dpi         = 100) {
  
  all_dates <- format(seq(as.Date("1960-01-01"),
                          as.Date("2024-06-01"),
                          by = "month"), "%Y-%m")
  
  available_depths <- unique(terra::depth(terra::rast(nc_path)))
  n_depths         <- length(available_depths)
  
  depth_i <- which.min(abs(available_depths - depth_target))
  cat("Depth investigated :", depth_target, "m → Depth available :",
      available_depths[depth_i], "m\n")
  cat("Number of timesteps :", length(all_dates), "\n")
  
  iso_exists <- any(bathy_df$depth <= -depth_target, na.rm = TRUE)
  
  frames_dir <- file.path(output_dir, paste0("frames_", depth_target, "m"))
  dir.create(frames_dir, recursive = TRUE, showWarnings = FALSE)
  
  pal <- rev(RColorBrewer::brewer.pal(11, "RdYlBu"))

  make_frame <- function(t_idx, nc_path, n_depths, depth_i, depth_target,
                          bathy_df, pal, all_dates, iso_exists, frames_dir) {
    
    layer_index <- (t_idx - 1) * n_depths + depth_i
    
    layer      <- terra::rast(nc_path, lyrs = layer_index)
    date_label <- all_dates[t_idx]
    
    p <- ggplot() +
      geom_spatraster(data = layer) +
      scale_fill_gradientn(colors   = pal,
                           na.value = "grey90",
                           name     = "variance %",
                           limits   = c(0, 100)) +
      annotation_borders("world", colour = "black", linewidth = 0.3) +
      coord_sf(xlim = c(-4.75, 13.75), ylim = c(48.55, 62.55))
    
    if (iso_exists) {
      p <- p +
        geom_contour(data      = bathy_df,
                     aes(x = lon, y = lat, z = depth),
                     breaks    = -depth_target,
                     na.rm     = TRUE,
                     color     = "red",
                     linewidth = 0.6)
    }
    
    p <- p +
      labs(title    = paste0("Temperature variance — ", depth_target, " m"),
           subtitle = date_label,
           x = NULL, y = NULL) +
      theme_minimal(base_size = 11) +
      theme(legend.position = "right",
            plot.title    = element_text(face = "bold", hjust = 0.5),
            plot.subtitle = element_text(hjust = 0.5, color = "grey50", size = 12))
    
    fname <- file.path(frames_dir, sprintf("frame_%04d_%s.png", t_idx, date_label))
    ggsave(fname, plot = p, width = 8, height = 7, dpi = dpi)
    return(fname)
  }
  
  plan(multisession, workers = n_workers)
  cat("Genrating", length(all_dates), "frames on", n_workers, "workers...\n")
  
  frame_files <- future_map_chr(
    seq_along(all_dates),
    ~ make_frame(.x,
                 nc_path      = nc_path,
                 n_depths     = n_depths,
                 depth_i      = depth_i,
                 depth_target = depth_target,
                 bathy_df     = bathy_df,
                 pal          = pal,
                 all_dates    = all_dates,
                 iso_exists   = iso_exists,
                 frames_dir   = frames_dir),
    .progress = TRUE
  )
  
  plan(sequential)
  cat("Frames generated ✓\n")
}


In [ ]:
nc_path <- "cmems_obs-ins_glo_phy-temp-sal_my_cora-oa_P1M_TEMP_PCTVAR_4.50W-13.50E_48.60N-62.50N_1.00-5500.00m_1960-01-01-2024-06-01.nc"

for (d in target_depths) {
  cat("\n========================================\n")
  cat("Traitement profondeur :", d, "m\n")
  cat("========================================\n")
  
  generate_temp_var_frames(depth_target = d, nc_path = nc_path, n_workers = 10)
  
  plan(sequential)                    
  gc()                                
  terra::terraOptions(memfrac = 0.5)  
  Sys.sleep(25)                       
  
  cat("Memory cleaned ✓\n")
}
cat("\nAll frames generated ✓\n")

**Function to create the GIFs (one per depth)**

In [ ]:
generate_gifs_from_frames <- function(base_dir = "/home/mathis/Desktop/temp_var_gif",
                               fps      = 6) {
  
  subdirs <- list.dirs(base_dir, full.names = TRUE, recursive = FALSE)
  subdirs <- subdirs[grepl("frames_\\d+m$", basename(subdirs))]
  
  cat("Subfolders found :", length(subdirs), "\n")
  
  for (subdir in subdirs) {
    
    depth_target <- as.integer(gsub("frames_(\\d+)m", "\\1", basename(subdir)))
    cat("\n--- Depth :", depth_target, "m ---\n")
    
    frame_files <- sort(list.files(subdir, pattern = "\\.png$", full.names = TRUE))
    cat("Frames found :", length(frame_files), "\n")
    
    if (length(frame_files) == 0) {
      cat("No frame found, depth is ignored.\n")
      next
    }
    
    gif_path <- file.path(base_dir,
                           paste0("temp_", depth_target, "m_1960-2024.gif"))
    
    cat("Assembling GIF...\n")
    
    gifski(frame_files,
           gif_file = gif_path,
           delay = 1 / fps,
           loop = 0)        
    
    cat("GIF saved :", gif_path, "\n")
    gc()
  }
  
  cat("\nAll GIFs are generated ✓\n")
}

# --- Appel ---
generate_gifs_from_frames(base_dir = "/home/mathis/Desktop/temp_var_gif")